# Verifier: calibration on MedAESQA
Checks how often the Verifier agrees with MedAESQA's human labels (accuracy and error counts).

**Setup in Colab**
1. Open the key icon in the left sidebar, add a secret named `LLM_API_KEY` with your key, and turn on notebook access
   (use `GEMINI_API_KEY` instead if you are using Gemini).
2. Run the cells in order. When asked, click **Choose Files** and select `medaesqa_v1.json`
   (download it from https://osf.io/ydbzq).

In [1]:
%pip install -q google-genai openai pydantic python-dotenv scikit-learn pandas numpy

Settings

In [2]:
import os, re, json, time, threading
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from pydantic import BaseModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

try:
    from dotenv import load_dotenv
    load_dotenv()   # reads a local .env file if there is one (VS Code / local use)
except ImportError:
    pass

PROVIDER = os.getenv("LLM_PROVIDER") or "openai"        # "openai" (OpenRouter etc.) or "gemini"
MODEL = os.getenv("LLM_MODEL") or "stealth/crouton-alpha"
BASE_URL = os.getenv("LLM_BASE_URL") or "https://openrouter.ai/api/v1"
TEMPERATURE = 0      # set to None if the model rejects it
WORKERS = 2          # parallel calls; lower it if you see rate-limit errors
MAX_CALLS = 500      # safety stop so a bug cannot burn through credits; raise it on purpose

def get_secret(name):
    value = os.getenv(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        raise RuntimeError(f"Missing {name}. Put it in a .env file (see .env.example), "
                           "set it as an environment variable, or add it to Colab Secrets.")
    return value

if PROVIDER == "gemini":
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=get_secret("GEMINI_API_KEY"))
else:
    from openai import OpenAI
    client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=BASE_URL, timeout=60, max_retries=0)

The Verifier (same rules as the pipeline notebook)

In [15]:
class Verdict(BaseModel):
    # Used only to steer Gemini's structured output.
    unsupported_claims: list[str]
    omissions: list[str]
    reading_level_ok: bool
    faithful: bool


KEYS = ["faithful", "unsupported_claims", "omissions", "reading_level_ok"]

INTRO = """You are a clinical faithfulness verifier. Judge the TEXT against the SOURCE only.

The SOURCE is a set of short evidence excerpts from the papers the TEXT cites.

"""

RULES = """Work through these steps silently, then return only the JSON object.

1. Use only the SOURCE. A statement that is true in general medicine but is not in the SOURCE
   is not supported. Do not assume the SOURCE is complete: a claim it does not state is "not
   stated", which is different from "wrong".

2. Split the TEXT into claims. A clinical claim tells the reader something about a diagnosis,
   test result, number, treatment, drug, dose, cause, effect, risk, or what to do. Ignore
   framing words ("Additionally", "It is important to note"), generic reassurance, advice such
   as "ask your doctor", and plain explanations of a term that add no new clinical fact.

3. Check every clinical claim against the SOURCE:
     supported = the SOURCE states it, entails it, or says the same thing in different words.
               Hedging must match: a SOURCE that says "may" or reports mixed results supports
               only a hedged claim.
   contradicted = the SOURCE says something different or opposite: the wrong direction
               (increase vs decrease), a wrong number, unit, dose, frequency or duration, an
               effect that continues where the SOURCE says it stops, a definite statement where
               the SOURCE is uncertain, an effect where the SOURCE found none, found it not
               significant, or found it only for something else, or the opposite of a negative
               finding ("does not", "no effect").
   not stated = the SOURCE says nothing about it.
   Make sure the SOURCE is about the same thing the claim is about: the same condition, drug,
   test, side of the body and population, and the same relationship between them. Do not carry
   a fact over from a similar thing (one drug to another, type 1 to type 2 diabetes, adults to
   children). Sharing a topic is not support. Words like causes, prevents, cures, reduces risk,
   is safe, does not cause, or is not needed need an explicit statement in the SOURCE. Do not
   decide that a value is normal or abnormal from the number alone; use what the SOURCE says.

4. unsupported_claims: one entry for every clinical claim in the TEXT that is contradicted or
   not stated. Each entry is one short sentence that names the specific detail so it can be
   found in the TEXT, says whether the SOURCE contradicts it or does not state it, and ends
   with a few words on the fix (remove it, correct it to the SOURCE value, or soften it).
   Style:
   "The dose is incorrect: the SOURCE says 500 mg twice daily, so correct it."
   "The claim that the drug improves survival is not stated in the SOURCE, so remove it."
   Only list problems that really are in the TEXT.

5. omissions: list a fact only if all three are true: the SOURCE states it, the TEXT does not
   already convey it (even in different words), and leaving it out would change what the
   patient understands or does next (a main diagnosis, a medication change, a result the SOURCE
   calls abnormal or critical, follow-up, a warning). Do not list facts just for completeness.
   Each entry is one short sentence. Return an empty list if nothing important is missing.

6. reading_level_ok: true if the wording fits roughly an 8th-grade reading level.

7. faithful: true only if unsupported_claims is empty.

Return exactly one JSON object with exactly these four keys and no others, and no text outside it:
{"faithful": true or false, "unsupported_claims": ["..."], "omissions": ["..."], "reading_level_ok": true or false}"""


def build_prompt(source, text):
    return INTRO + "SOURCE:\n" + source + "\n\nTEXT TO AUDIT:\n" + text + "\n\n" + RULES


def parse_report(text):
    """Accept the reply only if it is exactly the four-key object; otherwise raise ValueError."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (text or "").strip())
    d = json.loads(text)
    if not isinstance(d, dict) or set(d) != set(KEYS):
        raise ValueError(f"wrong keys: {sorted(d) if isinstance(d, dict) else type(d).__name__}")
    if not isinstance(d["faithful"], bool) or not isinstance(d["reading_level_ok"], bool):
        raise ValueError("faithful and reading_level_ok must be true/false")
    for k in ("unsupported_claims", "omissions"):
        if not isinstance(d[k], list) or not all(isinstance(x, str) for x in d[k]):
            raise ValueError(f"{k} must be a list of strings")
    return {k: d[k] for k in KEYS}


stats = {"calls": 0, "format_errors": 0, "tokens_in": 0, "tokens_out": 0}

def ask_llm(prompt):
    temp = {} if TEMPERATURE is None else {"temperature": TEMPERATURE}
    if PROVIDER == "gemini":
        cfg = types.GenerateContentConfig(response_mime_type="application/json",
                                          response_schema=Verdict, **temp)
        return client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}, **temp)
    if getattr(r, "usage", None):
        stats["tokens_in"] += r.usage.prompt_tokens or 0
        stats["tokens_out"] += r.usage.completion_tokens or 0
    return r.choices[0].message.content or ""


QUOTA_ERRORS = ("PerDay", "per-day", "insufficient_quota", "Error code: 402")

def judge(source, text):
    prompt = build_prompt(source, text)
    last_err = None
    for attempt in range(4):
        if stats["calls"] >= MAX_CALLS:
            raise RuntimeError(f"Call budget of {MAX_CALLS} reached. Raise MAX_CALLS if that is intended.")
        try:
            stats["calls"] += 1
            return parse_report(ask_llm(prompt))
        except ValueError as e:          # wrong shape or invalid JSON: ask again
            stats["format_errors"] += 1
            last_err = e
        except Exception as e:           # API error
            if any(s in str(e) for s in QUOTA_ERRORS):
                raise RuntimeError("Out of quota or credits. Switch model or wait.") from e
            print(f"attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt * 3)
    raise RuntimeError(f"judge failed after retries: {last_err}")

Quick check. Expect `faithful: false` with the "once a day" claim listed.

In [6]:
print(json.dumps(judge("Metformin 500 mg twice daily for type 2 diabetes.",
                       "You take a diabetes pill called metformin once a day."), indent=2))

{
  "faithful": false,
  "unsupported_claims": [
    "The frequency 'once a day' is incorrect: the SOURCE says twice daily, so correct it."
  ],
  "omissions": [],
  "reading_level_ok": true
}


MedAESQA data. Human label: 1 if a cited paper supports the sentence and none contradict it, otherwise 0.

In [12]:
DATA_PATH = os.getenv("MEDAESQA_PATH") or "data/medaesqa_v1.json"

if not os.path.exists(DATA_PATH):
    try:                                   # in Colab, fall back to the upload button
        from google.colab import files
        DATA_PATH = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"{DATA_PATH} not found. Download medaesqa_v1.json from https://osf.io/ydbzq "
            "and put it there, or set MEDAESQA_PATH in your .env file.")

with open(DATA_PATH) as f:
    data = json.load(f)

rows = []
for q in data:
    for mid, ans in q["machine_generated_answers"].items():
        for s in ans["answer_sentences"]:
            ca = s["citation_assessment"]
            if not ca:
                continue
            ev = [c["evidence_support"] for c in ca if c["evidence_support"]]
            if not ev:
                continue
            rels = {c["evidence_relation"] for c in ca}
            rows.append({
                "question_id": q["question_id"],
                "source_text": "\n".join(ev),
                "answer_text": re.sub(r"\s*\[[\d,\s]+\]", "", s["answer_sentence"]).strip(),
                "rels": "|".join(sorted(rels)),
                "human_label": int("supporting" in rels and "contradicting" not in rels),
            })

df = (pd.DataFrame(rows)
        .drop_duplicates(subset=["question_id", "source_text", "answer_text"])
        .reset_index(drop=True))

# split by question so no question appears in both halves
train_q, test_q = train_test_split(df["question_id"].unique(), test_size=0.5, random_state=42)
train_df = df[df["question_id"].isin(train_q)].reset_index(drop=True)
test_df = df[df["question_id"].isin(test_q)].reset_index(drop=True)

print(len(df), "rows |", len(train_df), "train,", len(test_df), "test")
print("train labels:", train_df.human_label.value_counts().to_dict())
print("test labels: ", test_df.human_label.value_counts().to_dict())

Saving medaesqa_v1.json to medaesqa_v1 (1).json
3526 rows | 1857 train, 1669 test
train labels: {1: 1693, 0: 164}
test labels:  {1: 1468, 0: 201}


Calibration on the training half. Seed fixed for testing purposes.

In [17]:
def accuracy_range(y, p, n=2000, seed=0):
    """90% range for accuracy."""
    rng = np.random.default_rng(seed)
    accs = [np.mean(y[i] == p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(accs, [5, 95])


def calibrate(df, workers=WORKERS):
    """df needs: source_text, answer_text, human_label (0/1)"""
    rows = list(df.itertuples())
    done, errors, lock = [0], [], threading.Lock()

    def run(r):
        try:
            v = judge(r.source_text, r.answer_text)
        except RuntimeError as e:        # quota, call budget or repeated failure: keep what finished
            with lock:
                errors.append(str(e))
            return None
        with lock:
            done[0] += 1
            if done[0] % 20 == 0:
                print(f"{done[0]}/{len(rows)} done")
        return v

    with ThreadPoolExecutor(max_workers=workers) as ex:
        verdicts = list(ex.map(run, rows))

    ok = [i for i, v in enumerate(verdicts) if v is not None]
    if errors:
        print(f"WARNING: {len(errors)} of {len(rows)} rows failed, so this run is incomplete and not "
              f"comparable with a full run. First error: {errors[0]}")
    if not ok:
        raise RuntimeError("No rows were scored. See the error above.")
    out = df.iloc[ok].assign(verifier_label=[int(verdicts[i]["faithful"]) for i in ok],
                             verdict=[json.dumps(verdicts[i]) for i in ok])
    y = out["human_label"].astype(int).values
    p = out["verifier_label"].values
    lo, hi = accuracy_range(y, p)
    print(f"rows scored: {len(out)} of {len(rows)}")
    print(f"accuracy {accuracy_score(y, p):.3f} (90% range {lo:.2f} to {hi:.2f})")
    print(f"false pos {int(((p == 1) & (y == 0)).sum())} | false neg {int(((p == 0) & (y == 1)).sum())}")
    print("confusion matrix [[TN, FP], [FN, TP]]:", confusion_matrix(y, p, labels=[0, 1]).tolist())
    print(f"format check: {stats['format_errors']} replies rejected for wrong shape out of {stats['calls']} calls")
    print(f"tokens in {stats['tokens_in']:,} | out {stats['tokens_out']:,}")
    return out


N = 50

pos = train_df[train_df.human_label == 1].sample(N, random_state=1)
neg = train_df[train_df.human_label == 0].sample(N, random_state=1)
sample = pd.concat([pos, neg]).sample(frac=1, random_state=1)
results = calibrate(sample)

20/100 done
40/100 done
60/100 done
80/100 done
100/100 done
rows scored: 100 of 100
accuracy 0.670 (90% range 0.59 to 0.75)
false pos 8 | false neg 25
confusion matrix [[TN, FP], [FN, TP]]: [[42, 8], [25, 25]]
format check: 0 replies rejected for wrong shape out of 140 calls
tokens in 134,082 | out 9,652


Where the disagreements are

In [18]:
pd.set_option("display.max_colwidth", 250)

def show(rows):
    for r in rows.itertuples():
        print("SOURCE:", r.source_text[:400])
        print("TEXT:  ", r.answer_text)
        print("HUMAN:", r.human_label, "| VERIFIER:", r.verifier_label)
        print("OUTPUT:", r.verdict)
        print("-" * 60)

wrong = results[results.human_label != results.verifier_label]
print(len(wrong), "disagreements")
show(wrong.head(6))

33 disagreements
SOURCE: In humans, meiotic chromosome segregation errors increase dramatically as women age, but the molecular defects responsible are largely unknown
Our data provide the first demonstration in any organism that, when meiotic cohesion begins intact, the aging process can weaken it sufficiently and cause missegregation of recombinant chromosomes.
This specific type of error is estimated to account for ove
TEXT:   Increased maternal age is a significant risk factor, likely due to age-related deterioration of cohesion proteins that maintain chromosome structure.
HUMAN: 0 | VERIFIER: 1
OUTPUT: {"faithful": true, "unsupported_claims": [], "omissions": [], "reading_level_ok": true}
------------------------------------------------------------
SOURCE: However, ocular surface showed self-healing in 20% (n = 3/15) of the rabbits with a score of 1.67 ± 0.34 in four months irrespective of similar alkali injury.
Our study shows the essentiality to consider the time required for su

In [10]:
# Pass rate by the type of human judgment on the cited papers
print(results.groupby("rels").agg(
    rows=("human_label", "size"),
    human_label=("human_label", "mean"),
    verifier_pass=("verifier_label", "mean"),
).sort_values("rows", ascending=False).round(2))

                                     rows  human_label  verifier_pass
rels                                                                 
supporting                             12          1.0           0.33
neutral                                 8          0.0           0.00
contradicting|supporting                4          0.0           0.00
not relevant|supporting                 3          1.0           0.00
neutral|supporting                      3          1.0           0.00
contradicting|neutral|supporting        3          0.0           0.00
contradicting                           2          0.0           0.00
neutral|not relevant                    2          0.0           0.00
contradicting|neutral|not relevant      1          0.0           0.00
invalid citation|supporting             1          1.0           0.00
invalid citation|neutral|supporting     1          1.0           1.00
